In [ ]:

# Cell 1: Check environment and GPU

import sys
import torch
import pandas as pd
import numpy as np
import sklearn

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Please enable a GPU runtime if available.")


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
Pandas: 2.2.3
Scikit-learn: 1.6.1
GPU available: True
GPU: Tesla T4


In [ ]:

%pip install -q transformers datasets accelerate evaluate scikit-learn openpyxl shap


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00


In [ ]:

from google.colab import files

uploaded = files.upload()

print("Uploaded files:", list(uploaded.keys()))


Saving final_labels_SG2.xlsx to final_labels_SG2.xlsx
Uploaded files: ['final_labels_SG2.xlsx']


In [ ]:

import pandas as pd

FILE_PATH = "/content/final_labels_SG2.xlsx"

df = pd.read_excel(FILE_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nBias label counts:")
print(df["label_bias"].value_counts(dropna=False))

display(df.head())


Dataset shape: (3674, 8)

Columns:
['text', 'news_link', 'outlet', 'topic', 'type', 'label_bias', 'label_opinion', 'biased_words']

Bias label counts:
label_bias
Non-biased      1863
Biased          1810
No agreement       1
Name: count, dtype: int64


,text,news_link,outlet,topic,type,label_bias,label_opinion,biased_words
0,"""Orange Is the New Black"" star Yael Stone is r...",https://www.foxnews.com/entertainment/australi...,Fox News,environment,right,Non-biased,Entirely factual,[]
1,"""We have one beautiful law,"" Trump recently sa...",https://www.alternet.org/2020/06/law-and-order...,Alternet,gun control,left,Biased,Somewhat factual but also opinionated,"['bizarre', 'characteristically']"
2,"...immigrants as criminals and eugenics, all o...",https://www.nbcnews.com/news/latino/after-step...,MSNBC,white-nationalism,left,Biased,Expresses writer’s opinion,"['criminals', 'fringe', 'extreme']"
3,...we sounded the alarm in the early months of...,https://www.alternet.org/2019/07/fox-news-has-...,Alternet,white-nationalism,left,Biased,Somewhat factual but also opinionated,[]
4,[Black Lives Matter] is essentially a non-fals...,http://feedproxy.google.com/~r/breitbart/~3/-v...,Breitbart,marriage-equality,NaN,Biased,Expresses writer’s opinion,['cult']


In [ ]:

import pandas as pd
import numpy as np
import re

# Preserve the original dataset
df_original = df.copy()
df_clean = df.copy()

# Normalize whitespace only; do not remove meaningful words or punctuation
df_clean["text"] = (
    df_clean["text"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Standardize label whitespace
df_clean["label_bias"] = (
    df_clean["label_bias"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Keep only valid binary labels and non-empty text
valid_labels = {"Biased", "Non-biased"}

df_binary = df_clean[
    df_clean["text"].ne("") &
    df_clean["label_bias"].isin(valid_labels)
].copy()

print("Original rows:", len(df_original))
print("Rows with valid binary labels and text:", len(df_binary))
print("Rows removed:", len(df_original) - len(df_binary))

print("\nClass distribution:")
print(df_binary["label_bias"].value_counts())

print("\nMissing values:")
print(df_binary.isna().sum())

print("\nExact duplicate texts:", df_binary["text"].duplicated().sum())


Original rows: 3674
Rows with valid binary labels and text: 3673
Rows removed: 1

Class distribution:
label_bias
Non-biased    1863
Biased        1810
Name: count, dtype: int64

Missing values:
text                0
news_link          32
outlet              0
topic               0
type             1000
label_bias          0
label_opinion       0
biased_words        0
dtype: int64

Exact duplicate texts: 3


In [ ]:

# Find all repeated text groups
duplicate_rows = df_binary[
    df_binary.duplicated(subset=["text"], keep=False)
].sort_values("text")

print("Rows belonging to repeated-text groups:", len(duplicate_rows))

# Count distinct labels assigned to each normalized text
labels_per_text = df_binary.groupby("text")["label_bias"].nunique()

conflicting_texts = labels_per_text[labels_per_text > 1]

print("Texts with conflicting labels:", len(conflicting_texts))

if len(conflicting_texts) > 0:
    display(
        df_binary[df_binary["text"].isin(conflicting_texts.index)][
            ["text", "news_link", "outlet", "label_bias"]
        ].head(20)
    )

if len(duplicate_rows) > 0:
    print("\nSample repeated texts:")
    display(
        duplicate_rows[
            ["text", "news_link", "outlet", "label_bias"]
        ].head(10)
    )


Rows belonging to repeated-text groups: 6
Texts with conflicting labels: 1


,text,news_link,outlet,label_bias
2836,The police lynching of George Floyd provoked w...,https://www.alternet.org/2020/06/law-and-order...,Alternet,Biased
2837,The police lynching of George Floyd provoked w...,https://www.alternet.org/2020/06/law-and-order...,Alternet,Non-biased



Sample repeated texts:


,text,news_link,outlet,label_bias
790,Demagogue that he is and that he’s toyed with ...,https://www.alternet.org/2020/06/the-die-is-ca...,Alternet,Biased
791,Demagogue that he is and that he’s toyed with ...,https://www.alternet.org/2020/06/the-die-is-ca...,Alternet,Biased
873,"Donald Trump, by design, is a chaos monster wh...",https://www.alternet.org/2020/05/trump-is-desp...,Alternet,Biased
874,"Donald Trump, by design, is a chaos monster wh...",https://feeds.feedblitz.com/~/625454580/0/alte...,Alternet,Biased
2836,The police lynching of George Floyd provoked w...,https://www.alternet.org/2020/06/law-and-order...,Alternet,Biased
2837,The police lynching of George Floyd provoked w...,https://www.alternet.org/2020/06/law-and-order...,Alternet,Non-biased


In [ ]:

# Cell 7: Remove conflicting and repeated text groups safely

# Find text groups that have more than one distinct label
labels_per_text = df_binary.groupby("text")["label_bias"].nunique()
conflicting_texts = set(labels_per_text[labels_per_text > 1].index)

# Save all rows excluded due to conflicting labels
df_conflicts = df_binary[
    df_binary["text"].isin(conflicting_texts)
].copy()

# Exclude conflicting text groups
df_model = df_binary[
    ~df_binary["text"].isin(conflicting_texts)
].copy()

# Remove duplicate texts, keeping one representative row
df_duplicates_removed = df_model[
    df_model.duplicated(subset=["text"], keep="first")
].copy()

df_model = df_model.drop_duplicates(
    subset=["text"],
    keep="first"
).copy()

df_model = df_model.reset_index(drop=True)

print("Rows before duplicate handling:", len(df_binary))
print("Conflicting rows excluded:", len(df_conflicts))
print("Redundant duplicate rows removed:", len(df_duplicates_removed))
print("Rows after cleaning:", len(df_model))

print("\nFinal label counts:")
print(df_model["label_bias"].value_counts())

print("\nDuplicate check:")
print("Repeated texts remaining:", df_model["text"].duplicated().sum())

# Save records for auditability
df_conflicts.to_csv("/content/conflicting_label_examples.csv", index=False)
df_duplicates_removed.to_csv("/content/redundant_duplicate_examples.csv", index=False)
df_model.to_csv("/content/cleaned_news_bias_dataset.csv", index=False)


Rows before duplicate handling: 3673
Conflicting rows excluded: 2
Redundant duplicate rows removed: 2
Rows after cleaning: 3669

Final label counts:
label_bias
Non-biased    1862
Biased        1807
Name: count, dtype: int64

Duplicate check:
Repeated texts remaining: 0


In [ ]:

# Safety cell: preserve original data

import os
import shutil

# Save the original dataframe without cleaning or filtering
df_original.to_excel(
    "/content/original_dataset_backup.xlsx",
    index=False
)

# Also preserve the exact uploaded Excel file
if os.path.exists("/content/final_labels_SG2.xlsx"):
    shutil.copy2(
        "/content/final_labels_SG2.xlsx",
        "/content/final_labels_SG2_original.xlsx"
    )

print("Original dataset backup saved.")
print("Original row count:", len(df_original))
print("Original dataframe remains unchanged:", len(df_original) == 3674)


Original dataset backup saved.
Original row count: 3674
Original dataframe remains unchanged: True


In [ ]:

# Step 7: Safe duplicate handling with an audit trail

# Start from the binary-labelled copy created earlier.
# df_original remains unchanged.
df_binary = df_original.copy()

df_binary["text"] = (
    df_binary["text"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df_binary["label_bias"] = (
    df_binary["label_bias"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Record the one unresolved-label row instead of losing it.
valid_labels = {"Biased", "Non-biased"}

excluded_unresolved = df_binary[
    ~df_binary["label_bias"].isin(valid_labels) |
    df_binary["text"].eq("")
].copy()

excluded_unresolved["exclusion_reason"] = "Unresolved label or empty text"

# Binary examples for the modelling copy
df_binary = df_binary[
    df_binary["label_bias"].isin(valid_labels) &
    df_binary["text"].ne("")
].copy()

# Identify texts with contradictory labels
label_counts = df_binary.groupby("text")["label_bias"].nunique()
conflicting_texts = set(label_counts[label_counts > 1].index)

excluded_conflicts = df_binary[
    df_binary["text"].isin(conflicting_texts)
].copy()

excluded_conflicts["exclusion_reason"] = "Conflicting labels for identical text"

# Remove conflicting text groups from this training copy
df_model = df_binary[
    ~df_binary["text"].isin(conflicting_texts)
].copy()

# Save redundant duplicate rows before deduplicating
duplicate_mask = df_model.duplicated(subset=["text"], keep="first")

excluded_duplicates = df_model[duplicate_mask].copy()
excluded_duplicates["exclusion_reason"] = (
    "Repeated identical text with same label; one copy retained"
)

df_model = df_model.drop_duplicates(
    subset=["text"],
    keep="first"
).reset_index(drop=True)

# Save every excluded row in one audit file
audit_df = pd.concat(
    [excluded_unresolved, excluded_conflicts, excluded_duplicates],
    ignore_index=True
)

audit_df.to_csv("/content/excluded_rows_audit.csv", index=False)
df_model.to_csv("/content/cleaned_news_bias_dataset.csv", index=False)

print("Original rows:", len(df_original))
print("Rows in modelling copy:", len(df_model))
print("Unresolved/empty rows excluded:", len(excluded_unresolved))
print("Conflicting-label rows excluded:", len(excluded_conflicts))
print("Redundant duplicate rows excluded:", len(excluded_duplicates))
print("Audit records saved:", len(audit_df))

print("\nFinal modelling class counts:")
print(df_model["label_bias"].value_counts())

print("\nOriginal dataframe still intact:", len(df_original) == 3674)


Original rows: 3674
Rows in modelling copy: 3669
Unresolved/empty rows excluded: 1
Conflicting-label rows excluded: 2
Redundant duplicate rows excluded: 2
Audit records saved: 5

Final modelling class counts:
label_bias
Non-biased    1862
Biased        1807
Name: count, dtype: int64

Original dataframe still intact: True


In [ ]:
import os
import pandas as pd

cleaned_path = "/content/cleaned_news_bias_dataset.csv"
audit_path = "/content/excluded_rows_audit.csv"
backup_path = "/content/original_dataset_backup.xlsx"

for path in [cleaned_path, audit_path, backup_path]:
    print(f"{os.path.basename(path)}: {'Found' if os.path.exists(path) else 'MISSING'}")

df_clean = pd.read_csv(cleaned_path)

print("\nCleaned dataset shape:", df_clean.shape)
print("\nClass counts:")
print(df_clean["label_bias"].value_counts())

print("\nMissing text values:", df_clean["text"].isna().sum())
print("Missing labels:", df_clean["label_bias"].isna().sum())
print("Duplicate texts:", df_clean["text"].duplicated().sum())

cleaned_news_bias_dataset.csv: Found
excluded_rows_audit.csv: Found
original_dataset_backup.xlsx: Found

Cleaned dataset shape: (3669, 8)

Class counts:
label_bias
Non-biased    1862
Biased        1807
Name: count, dtype: int64

Missing text values: 0
Missing labels: 0
Duplicate texts: 0


In [ ]:
import pandas as pd
import re
import hashlib
from urllib.parse import urlparse, urlunparse

# Load the cleaned modelling dataset
df_clean = pd.read_csv("/content/cleaned_news_bias_dataset.csv")

def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def normalize_url(url):
    """Normalize URL formatting while retaining the article path."""
    if pd.isna(url) or not str(url).strip():
        return None

    url = str(url).strip()

    # Make URLs parseable even if the scheme is missing
    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url

    parsed = urlparse(url)

    host = (parsed.hostname or "").lower()
    if host.startswith("www."):
        host = host[4:]

    path = re.sub(r"/+", "/", parsed.path or "/").rstrip("/")

    # Ignore query parameters and fragments; they often track referrals
    canonical_url = host + path

    return canonical_url if host else None

# Add normalized URLs and stable article-group IDs
df_clean["normalized_url"] = df_clean["news_link"].apply(normalize_url)

def make_group_id(row):
    if row["normalized_url"]:
        return "url:" + row["normalized_url"]

    # If URL is missing, use a hash of the normalized article text
    text = normalize_text(row["text"])
    text_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()
    return "text:" + text_hash

df_clean["group_id"] = df_clean.apply(make_group_id, axis=1)

# Inspect group sizes and label consistency
group_stats = (
    df_clean.groupby("group_id")
    .agg(
        rows=("text", "size"),
        unique_labels=("label_bias", "nunique"),
        labels=("label_bias", lambda x: ", ".join(sorted(set(x))))
    )
    .reset_index()
)

print("Rows:", len(df_clean))
print("Unique article groups:", df_clean["group_id"].nunique())
print("Rows without a usable URL:", df_clean["normalized_url"].isna().sum())
print("Groups containing multiple rows:", (group_stats["rows"] > 1).sum())
print(
    "Groups containing both labels:",
    (group_stats["unique_labels"] > 1).sum()
)

print("\nLargest groups:")
print(
    group_stats.sort_values("rows", ascending=False)
    .head(10)
    .to_string(index=False)
)

print("\nGroups containing both labels (if any):")
print(
    group_stats[group_stats["unique_labels"] > 1]
    .sort_values("rows", ascending=False)
    .head(20)
    .to_string(index=False)
)

# Save a copy with grouping metadata for later splitting
df_clean.to_csv("/content/news_bias_with_groups.csv", index=False)
print("\nSaved: /content/news_bias_with_groups.csv")

Rows: 3669
Unique article groups: 2109
Rows without a usable URL: 32
Groups containing multiple rows: 818
Groups containing both labels: 282

Largest groups:
                                                                                                                                          group_id  rows  unique_labels             labels
                                                                                              url:feedproxy.google.com/~r/breitbart/~3/MBTkPPu9uSo    15              2 Biased, Non-biased
                                                                                              url:feedproxy.google.com/~r/breitbart/~3/2RH0NVHEl6Q    15              1             Biased
                                                                                              url:feedproxy.google.com/~r/breitbart/~3/ei0CD392IU0    12              1             Biased
                                                                                              

In [ ]:
import pandas as pd

df_groups = pd.read_csv("/content/news_bias_with_groups.csv")

# Find groups containing more than one label
conflicting_ids = (
    df_groups.groupby("group_id")["label_bias"]
    .nunique()
    .loc[lambda x: x > 1]
    .index
)

print("Conflicting groups:", len(conflicting_ids))

# Inspect up to 10 groups, including all rows within each selected group
sample_ids = (
    df_groups[df_groups["group_id"].isin(conflicting_ids)]
    .groupby("group_id")
    .size()
    .sort_values(ascending=False)
    .head(10)
    .index
)

for group_id in sample_ids:
    group = df_groups[df_groups["group_id"] == group_id]

    print("\n" + "=" * 100)
    print("GROUP:", group_id)
    print("ROWS:", len(group))
    print("LABEL COUNTS:")
    print(group["label_bias"].value_counts().to_dict())

    for idx, row in group.iterrows():
        print(f"\nRow index: {idx}")
        print("Label:", row["label_bias"])
        print("Outlet:", row["outlet"])
        print("Topic:", row["topic"])
        print("URL:", row["news_link"])
        print("Text:", str(row["text"])[:500].replace("\n", " "))

Conflicting groups: 282

GROUP: url:feedproxy.google.com/~r/breitbart/~3/MBTkPPu9uSo
ROWS: 15
LABEL COUNTS:
{'Biased': 12, 'Non-biased': 3}

Row index: 385
Label: Biased
Outlet: Breitbart
Topic: gun control
URL: http://feedproxy.google.com/~r/breitbart/~3/MBTkPPu9uSo/
Text: As for Joe Biden, he’s made some truly bizarre comments about guns over his long political career, driven more by a desire to appease both gun-control advocates and gun-owning Democrat constituencies than any coherent philosophy.

Row index: 393
Label: Non-biased
Outlet: Breitbart
Topic: gun control
URL: http://feedproxy.google.com/~r/breitbart/~3/MBTkPPu9uSo/
Text: As Obamacare demonstrated, leftists are willing to take the electoral hit for imposing an unpopular policy if they believe that policy will socially engineer a more compliant American public in the years to come.

Row index: 1264
Label: Biased
Outlet: Breitbart
Topic: gun control
URL: http://feedproxy.google.com/~r/breitbart/~3/MBTkPPu9uSo/
Text: If the 

In [ ]:

import pandas as pd
import re

df_groups = pd.read_csv("/content/news_bias_with_groups.csv")

# Normalize whitespace for comparison without changing the original text
df_groups["normalized_text"] = (
    df_groups["text"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Check whether the same excerpt is assigned to multiple groups
text_group_counts = (
    df_groups.groupby("normalized_text")["group_id"]
    .nunique()
)

multi_group_texts = text_group_counts[text_group_counts > 1]

print("Total rows:", len(df_groups))
print("Unique normalized excerpts:", df_groups["normalized_text"].nunique())
print("Excerpts appearing in multiple groups:", len(multi_group_texts))

if len(multi_group_texts) > 0:
    examples = df_groups[
        df_groups["normalized_text"].isin(multi_group_texts.index)
    ][["normalized_text", "label_bias", "news_link", "group_id"]]

    print("\nExamples:")
    print(examples.head(10).to_string(index=False))
else:
    print("No identical normalized excerpts appear in multiple groups.")


Total rows: 3669
Unique normalized excerpts: 3669
Excerpts appearing in multiple groups: 0
No identical normalized excerpts appear in multiple groups.


In [ ]:

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df_groups = pd.read_csv("/content/news_bias_with_groups.csv")

# First: reserve approximately 15% of rows for testing.
split_1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=42
)

train_val_idx, test_idx = next(
    split_1.split(
        df_groups,
        y=df_groups["label_bias"],
        groups=df_groups["group_id"]
    )
)

train_val_df = df_groups.iloc[train_val_idx].copy()
test_df = df_groups.iloc[test_idx].copy()

# Second: reserve approximately 15% of the full dataset for validation.
# This is about 15 / 85 of the remaining data.
split_2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15 / 0.85,
    random_state=42
)

train_idx, val_idx = next(
    split_2.split(
        train_val_df,
        y=train_val_df["label_bias"],
        groups=train_val_df["group_id"]
    )
)

train_df = train_val_df.iloc[train_idx].copy()
val_df = train_val_df.iloc[val_idx].copy()

# Save the exact split for reuse by every model.
train_df.to_csv("/content/train_split.csv", index=False)
val_df.to_csv("/content/validation_split.csv", index=False)
test_df.to_csv("/content/test_split.csv", index=False)

# Report split sizes and class distributions.
for name, part in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}: {len(part)} rows ({len(part) / len(df_groups):.1%})")
    print(part["label_bias"].value_counts().to_dict())

# Confirm that no URL-based group crosses split boundaries.
train_groups = set(train_df["group_id"])
val_groups = set(val_df["group_id"])
test_groups = set(test_df["group_id"])

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

print("\nPASS: No group_id crosses train, validation, and test splits.")
print("Saved all three split files.")



Train: 2600 rows (70.9%)
{'Non-biased': 1310, 'Biased': 1290}

Validation: 541 rows (14.7%)
{'Non-biased': 294, 'Biased': 247}

Test: 528 rows (14.4%)
{'Biased': 270, 'Non-biased': 258}

PASS: No group_id crosses train, validation, and test splits.
Saved all three split files.


In [ ]:

import os
import torch
import pandas as pd

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

train_df = pd.read_csv("/content/train_split.csv")
val_df = pd.read_csv("/content/validation_split.csv")
test_df = pd.read_csv("/content/test_split.csv")

required_columns = {"text", "label_bias", "group_id"}

for name, part in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    missing_columns = required_columns - set(part.columns)
    assert not missing_columns, f"{name} is missing: {missing_columns}"
    assert part["text"].notna().all(), f"{name} has missing text"
    assert part["label_bias"].isin(
        ["Biased", "Non-biased"]
    ).all(), f"{name} has unexpected labels"

print("\nAll split files loaded and validated.")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4

All split files loaded and validated.
Train: (2600, 10)
Validation: (541, 10)
Test: (528, 10)


In [ ]:

%pip install -q transformers datasets accelerate scikit-learn evaluate sentencepiece


In [ ]:

import os
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    f1_score
)

set_seed(42)

MODEL_NAME = "FacebookAI/roberta-base"
OUTPUT_DIR = "/content/roberta_bias_model"

LABEL2ID = {
    "Non-biased": 0,
    "Biased": 1
}
ID2LABEL = {
    0: "Non-biased",
    1: "Biased"
}

print("Model:", MODEL_NAME)
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")
print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))


Model: FacebookAI/roberta-base
Device: GPU
Training rows: 2600
Validation rows: 541
Test rows: 528


In [ ]:

MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def prepare_dataset(frame):
    frame = frame[["text", "label_bias"]].copy()
    frame["text"] = frame["text"].astype(str)
    frame["labels"] = frame["label_bias"].map(LABEL2ID)

    assert frame["labels"].notna().all(), "Unexpected label found"

    dataset = Dataset.from_pandas(
        frame[["text", "labels"]],
        preserve_index=False
    )

    def tokenize_batch(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_LENGTH
        )

    return dataset.map(
        tokenize_batch,
        batched=True,
        remove_columns=["text"]
    )

train_ds = prepare_dataset(train_df)
val_ds = prepare_dataset(val_df)
test_ds = prepare_dataset(test_df)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_ds)
print(val_ds)
print(test_ds)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/2600 [00:00<?, ? examples/s]

Map:   0%|          | 0/541 [00:00<?, ? examples/s]

Map:   0%|          | 0/528 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 2600
})
Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 541
})
Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 528
})


In [ ]:

import transformers
import inspect
from transformers import TrainingArguments

print("Transformers version:", transformers.__version__)
print(inspect.signature(TrainingArguments.__init__))


Transformers version: 5.20.1
(self, output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = False, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_co

In [ ]:

import torch
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Evaluate and save after each epoch
    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=50,

    # Optimization
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_steps=100,

    # Use mixed precision on the Colab T4 GPU
    fp16=torch.cuda.is_available(),

    # Select the best checkpoint using validation macro-F1
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none",
    seed=42,
    data_seed=42
)

print("TrainingArguments created successfully.")
print("Evaluation strategy:", training_args.eval_strategy)
print("Warmup steps:", training_args.warmup_steps)
print("Best-model metric:", training_args.metric_for_best_model)


TrainingArguments created successfully.
Evaluation strategy: IntervalStrategy.EPOCH
Warmup steps: 100
Best-model metric: macro_f1


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

train_result = trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("\nTraining finished.")
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation macro-F1:", trainer.state.best_metric)
print("Saved model to:", OUTPUT_DIR)

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall
1,1.021792,0.476054,0.787431,0.778944,0.802302,0.776268
2,0.792180,0.443946,0.802218,0.795221,0.815139,0.792138
3,0.556394,0.501943,0.822551,0.821083,0.821311,0.820878
4,0.389178,0.636037,0.818854,0.817356,0.817581,0.817153


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training finished.
Best checkpoint: /content/roberta_bias_model/checkpoint-489
Best validation macro-F1: 0.8210830921868542
Saved model to: /content/roberta_bias_model


In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# Predict on the held-out test dataset
test_output = trainer.predict(test_ds)

test_logits = test_output.predictions
test_labels = test_output.label_ids
test_predictions = np.argmax(test_logits, axis=-1)

print("RO BERTa BASE — TEST RESULTS\n")

print(
    classification_report(
        test_labels,
        test_predictions,
        labels=[0, 1],
        target_names=["Non-biased", "Biased"],
        digits=4,
        zero_division=0
    )
)

print("Confusion matrix (true labels = rows, predictions = columns):")
print(confusion_matrix(test_labels, test_predictions, labels=[0, 1]))

roberta_results = {
    "model": "RoBERTa-base",
    "test_size": len(test_labels),
    "accuracy": accuracy_score(test_labels, test_predictions),
    "macro_f1": f1_score(
        test_labels,
        test_predictions,
        average="macro"
    )
}

print("\nSummary:")
print(pd.Series(roberta_results))

pd.DataFrame([roberta_results]).to_csv(
    "/content/roberta_test_results.csv",
    index=False
)

# Save predictions for later error analysis
test_predictions_df = test_df.copy()
test_predictions_df["true_label_id"] = test_labels
test_predictions_df["predicted_label_id"] = test_predictions
test_predictions_df["predicted_label"] = [
    "Biased" if p == 1 else "Non-biased"
    for p in test_predictions
]

test_predictions_df.to_csv(
    "/content/roberta_test_predictions.csv",
    index=False
)

print("\nSaved test metrics and predictions.")


RO BERTa BASE — TEST RESULTS

              precision    recall  f1-score   support

  Non-biased     0.8077    0.8140    0.8108       258
      Biased     0.8209    0.8148    0.8178       270

    accuracy                         0.8144       528
   macro avg     0.8143    0.8144    0.8143       528
weighted avg     0.8144    0.8144    0.8144       528

Confusion matrix (true labels = rows, predictions = columns):
[[210  48]
 [ 50 220]]

Summary:
model        RoBERTa-base
test_size             528
accuracy         0.814394
macro_f1         0.814327
dtype: object

Saved test metrics and predictions.


In [ ]:

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed
)

DEBERTA_NAME = "microsoft/deberta-v3-base"
DEBERTA_OUTPUT_DIR = "/content/deberta_bias_model"

set_seed(42)

deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_NAME)

print("DeBERTa tokenizer loaded.")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

DeBERTa tokenizer loaded.
GPU: Tesla T4


In [ ]:

from datasets import Dataset

def prepare_deberta_dataset(frame):
    work = frame[["text", "label_bias"]].copy()
    work["text"] = work["text"].astype(str)
    work["labels"] = work["label_bias"].map(LABEL2ID)

    assert work["labels"].notna().all()

    dataset = Dataset.from_pandas(
        work[["text", "labels"]],
        preserve_index=False
    )

    def tokenize_batch(batch):
        return deberta_tokenizer(
            batch["text"],
            truncation=True,
            max_length=256
        )

    return dataset.map(
        tokenize_batch,
        batched=True,
        remove_columns=["text"]
    )

deberta_train_ds = prepare_deberta_dataset(train_df)
deberta_val_ds = prepare_deberta_dataset(val_df)
deberta_test_ds = prepare_deberta_dataset(test_df)

deberta_collator = DataCollatorWithPadding(
    tokenizer=deberta_tokenizer
)

print("Train:", len(deberta_train_ds))
print("Validation:", len(deberta_val_ds))
print("Test:", len(deberta_test_ds))


Map:   0%|          | 0/2600 [00:00<?, ? examples/s]

Map:   0%|          | 0/541 [00:00<?, ? examples/s]

Map:   0%|          | 0/528 [00:00<?, ? examples/s]

Train: 2600
Validation: 541
Test: 528


In [ ]:

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

deberta_args = TrainingArguments(
    output_dir=DEBERTA_OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_steps=100,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=42,
    data_seed=42
)

deberta_trainer = Trainer(
    model=deberta_model,
    args=deberta_args,
    train_dataset=deberta_train_ds,
    eval_dataset=deberta_val_ds,
    processing_class=deberta_tokenizer,
    data_collator=deberta_collator,
    compute_metrics=compute_metrics
)

deberta_train_result = deberta_trainer.train()

deberta_trainer.save_model(DEBERTA_OUTPUT_DIR)
deberta_tokenizer.save_pretrained(DEBERTA_OUTPUT_DIR)

print("\nDeBERTa training finished.")
print("Best checkpoint:", deberta_trainer.state.best_model_checkpoint)
print("Best validation macro-F1:", deberta_trainer.state.best_metric)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

model.safetensors: reconstructing file:   0%|          |  0.00B /  371MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


ValueError: Attempting to unscale FP16 gradients.

In [ ]:

import torch
from transformers import TrainingArguments, Trainer

# Clear any leftover GPU memory from the failed run
if torch.cuda.is_available():
    torch.cuda.empty_cache()

deberta_args = TrainingArguments(
    output_dir=DEBERTA_OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_steps=100,

    # Important fix: disable mixed-precision FP16
    fp16=False,
    bf16=False,

    gradient_checkpointing=True,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=42,
    data_seed=42
)

print("DeBERTa training configuration updated.")
print("FP16:", deberta_args.fp16)
print("BF16:", deberta_args.bf16)
print("Training batch size:", deberta_args.per_device_train_batch_size)


DeBERTa training configuration updated.
FP16: False
BF16: False
Training batch size: 2


In [ ]:

from transformers import AutoModelForSequenceClassification

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

deberta_trainer = Trainer(
    model=deberta_model,
    args=deberta_args,
    train_dataset=deberta_train_ds,
    eval_dataset=deberta_val_ds,
    processing_class=deberta_tokenizer,
    data_collator=deberta_collator,
    compute_metrics=compute_metrics
)

deberta_train_result = deberta_trainer.train()

deberta_trainer.save_model(DEBERTA_OUTPUT_DIR)
deberta_tokenizer.save_pretrained(DEBERTA_OUTPUT_DIR)

print("\nDeBERTa training finished.")
print("Best checkpoint:", deberta_trainer.state.best_model_checkpoint)
print("Best validation macro-F1:", deberta_trainer.state.best_metric)


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall
1,0.000000,nan,0.543438,0.352096,0.271719,0.500000
2,0.000000,nan,0.543438,0.352096,0.271719,0.500000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "microsoft/deberta-v3-base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load a fresh model, not the model from the failed training run
check_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
check_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "Non-biased", 1: "Biased"},
    label2id={"Non-biased": 0, "Biased": 1},
).to(device)

# Check tokenizer/model vocabulary compatibility
embedding_count = check_model.get_input_embeddings().num_embeddings
print("Tokenizer vocabulary:", len(check_tokenizer))
print("Model embedding count:", embedding_count)
print("Padding token ID:", check_tokenizer.pad_token_id)

# Inspect a real batch from your existing dataset
check_model.eval()
batch = deberta_collator([deberta_train_ds[i] for i in range(4)])
batch = {k: v.to(device) for k, v in batch.items()}

print("Batch keys:", list(batch.keys()))
print("Labels:", batch["labels"].tolist())
print("Input IDs in range:",
      batch["input_ids"].min().item(),
      batch["input_ids"].max().item())

with torch.no_grad():
    output = check_model(**batch)

print("Initial loss:", output.loss.item())
print("Loss is finite:", torch.isfinite(output.loss).item())
print("Logits:\n", output.logits.cpu())
print("Logits are finite:",
      torch.isfinite(output.logits).all().item())


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

Tokenizer vocabulary: 128001
Model embedding count: 128100
Padding token ID: 0
Batch keys: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']
Labels: [0, 1, 1, 1]
Input IDs in range: 0 114541
Initial loss: 0.7243112325668335
Loss is finite: True
Logits:
 tensor([[-0.1265, -0.2433],
        [-0.1381, -0.2472],
        [-0.1243, -0.2478],
        [-0.1231, -0.2428]], dtype=torch.float16)
Logits are finite: True


In [ ]:

import torch
from transformers import AutoModelForSequenceClassification

# Start with a fresh model and explicitly use FP32
if torch.cuda.is_available():
    torch.cuda.empty_cache()

check_model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=2,
    id2label={0: "Non-biased", 1: "Biased"},
    label2id={"Non-biased": 0, "Biased": 1},
    torch_dtype=torch.float32,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
check_model = check_model.to(device=device, dtype=torch.float32)
check_model.train()

print("Default PyTorch dtype:", torch.get_default_dtype())
print("Model parameter dtype:", next(check_model.parameters()).dtype)

# Use a fresh batch from the existing training dataset
batch = deberta_collator([deberta_train_ds[i] for i in range(4)])
batch = {k: v.to(device) for k, v in batch.items()}

check_model.zero_grad(set_to_none=True)
output = check_model(**batch)

print("Logits dtype:", output.logits.dtype)
print("Loss:", output.loss.item())
print("Loss finite:", torch.isfinite(output.loss).item())

output.loss.backward()

bad_gradients = [
    name for name, param in check_model.named_parameters()
    if param.grad is not None
    and not torch.isfinite(param.grad).all().item()
]

print("Non-finite gradient tensors:", len(bad_gradients))
print("Examples:", bad_gradients[:10])


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

Default PyTorch dtype: torch.float32
Model parameter dtype: torch.float32
Logits dtype: torch.float32
Loss: 0.6542134284973145
Loss finite: True
Non-finite gradient tensors: 0
Examples: []


In [ ]:

import torch
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

DEBERTA_OUTPUT_DIR = "/content/deberta_bias_model_fp32"

deberta_args = TrainingArguments(
    output_dir=DEBERTA_OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,

    learning_rate=1e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_steps=100,

    fp16=False,
    bf16=False,
    gradient_checkpointing=False,
    optim="adamw_torch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=42,
    data_seed=42,
)

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    dtype=torch.float32,
)

deberta_trainer = Trainer(
    model=deberta_model,
    args=deberta_args,
    train_dataset=deberta_train_ds,
    eval_dataset=deberta_val_ds,
    processing_class=deberta_tokenizer,
    data_collator=deberta_collator,
    compute_metrics=compute_metrics,
)

deberta_train_result = deberta_trainer.train()

print("Best checkpoint:", deberta_trainer.state.best_model_checkpoint)
print("Best validation macro-F1:",
      deberta_trainer.state.best_metric)

deberta_trainer.save_model(DEBERTA_OUTPUT_DIR)
deberta_tokenizer.save_pretrained(DEBERTA_OUTPUT_DIR)


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall
1,4.359666,0.483725,0.776340,0.767758,0.789009,0.765416
2,3.091572,0.401199,0.824399,0.822212,0.824445,0.820960
3,2.580221,0.430977,0.829945,0.827607,0.830652,0.826062
4,2.012624,0.475731,0.828096,0.825337,0.829878,0.823391


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best checkpoint: /content/deberta_bias_model_fp32/checkpoint-489
Best validation macro-F1: 0.8276067499792181


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/deberta_bias_model_fp32/tokenizer_config.json',
 '/content/deberta_bias_model_fp32/tokenizer.json')

In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# Evaluate the best checkpoint selected using validation macro-F1
pred_output = deberta_trainer.predict(deberta_test_ds)

y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=-1)

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print("=== DEBERTA TEST RESULTS ===")
print(f"Test accuracy: {accuracy:.4%}")
print(f"Test macro-F1: {macro_f1:.4f}")

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["Non-biased", "Biased"],
    digits=4,
    zero_division=0,
))

print("Confusion matrix (true rows, predicted columns):")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

# Save predictions for later error analysis
test_frame = pd.read_csv("/content/test_split.csv")
test_frame["true_label_id"] = y_true
test_frame["predicted_label_id"] = y_pred
test_frame["true_label"] = test_frame["true_label_id"].map(
    {0: "Non-biased", 1: "Biased"}
)
test_frame["predicted_label"] = test_frame["predicted_label_id"].map(
    {0: "Non-biased", 1: "Biased"}
)
test_frame.to_csv("/content/deberta_test_predictions.csv", index=False)

pd.DataFrame([{
    "model": "DeBERTa-v3-base",
    "test_accuracy": accuracy,
    "test_macro_f1": macro_f1,
}]).to_csv("/content/deberta_test_results.csv", index=False)

print("\nSaved test predictions and metrics.")


=== DEBERTA TEST RESULTS ===
Test accuracy: 82.1970%
Test macro-F1: 0.8219

Classification report:
              precision    recall  f1-score   support

  Non-biased     0.8106    0.8295    0.8199       258
      Biased     0.8333    0.8148    0.8240       270

    accuracy                         0.8220       528
   macro avg     0.8220    0.8221    0.8219       528
weighted avg     0.8222    0.8220    0.8220       528

Confusion matrix (true rows, predicted columns):
[[214  44]
 [ 50 220]]

Saved test predictions and metrics.


In [ ]:
import os
import shutil
from transformers import AutoModelForSequenceClassification

export_dir = "/content/deberta_model_clean"

if os.path.exists(export_dir):
    shutil.rmtree(export_dir)

os.makedirs(export_dir, exist_ok=True)

# Export only the trained model and its configuration
deberta_trainer.save_model(export_dir)
deberta_tokenizer.save_pretrained(export_dir)

# Create a clean ZIP
zip_path = shutil.make_archive(
    "/content/deberta_model_clean",
    "zip",
    root_dir=export_dir
)

print("ZIP size (MB):", round(os.path.getsize(zip_path) / 1e6, 2))

from google.colab import files
files.download(zip_path)

NameError: name 'deberta_trainer' is not defined

In [ ]:
import os

paths = [
    "/content/deberta_bias_model_fp32",
    "/content/deberta_bias_model_fp32/checkpoint-489",
    "/content/deberta_news_bias_model.zip",
    "/content/deberta_test_results.csv",
]

for path in paths:
    print("FOUND " if os.path.exists(path) else "MISSING", path)

MISSING /content/deberta_bias_model_fp32
MISSING /content/deberta_bias_model_fp32/checkpoint-489
MISSING /content/deberta_news_bias_model.zip
MISSING /content/deberta_test_results.csv


In [ ]:
import os

drive_path = "/content/drive/MyDrive"

if os.path.exists(drive_path):
    print("Google Drive is accessible.")

    matches = []
    for root, dirs, files in os.walk(drive_path):
        # Skip common bulky folders
        dirs[:] = [d for d in dirs if d not in {
            ".cache", "node_modules", ".git"
        }]

        for name in files:
            if (
                "deberta_news_bias_model" in name.lower()
                or "deberta_bias_model" in name.lower()
                or name == "model.safetensors"
                or name == "pytorch_model.bin"
            ):
                matches.append(os.path.join(root, name))

    print("Possible backups:")
    print("\n".join(matches) if matches else "No matching backup found.")
else:
    print("Google Drive is not mounted in this runtime.")

Google Drive is not mounted in this runtime.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

root = "/content/drive/MyDrive"

for current, dirs, files in os.walk(root):
    dirs[:] = [
        d for d in dirs
        if d not in {".git", "node_modules", ".cache"}
    ]

    for name in files:
        if any(key in name.lower() for key in [
            "deberta_news_bias_model",
            "deberta_bias_model"
        ]):
            print(os.path.join(current, name))

In [ ]:
import os

root = "/content/drive/MyDrive"

for current, dirs, files in os.walk(root):
    dirs[:] = [d for d in dirs if d not in {
        ".git", "node_modules", ".cache"
    }]

    for name in files:
        if name.lower().endswith((
            ".zip", ".safetensors", ".bin",
            ".xlsx", ".csv"
        )):
            print(os.path.join(current, name))

/content/drive/MyDrive/Airbases_Errors.csv
/content/drive/MyDrive/Classroom/PSDL 25-26 A,B,C/UCE2024431_Deepfake_detection.zip
/content/drive/MyDrive/Classroom/AIML/news-bias-detection-main (2).zip
/content/drive/MyDrive/Classroom/AIML/news-bias-detection-main UCE2024431 A2.zip
/content/drive/MyDrive/NewsBais1/newsbias_roberta_best/training_args.bin
/content/drive/MyDrive/NewsBais1/newsbias_roberta_best/model.safetensors


In [ ]:
import zipfile
import os

archives = [
    "/content/drive/MyDrive/Classroom/AIML/news-bias-detection-main (2).zip",
    "/content/drive/MyDrive/Classroom/AIML/news-bias-detection-main UCE2024431 A2.zip",
]

for archive in archives:
    print("\nARCHIVE:", os.path.basename(archive))

    if not os.path.exists(archive):
        print("File not found")
        continue

    with zipfile.ZipFile(archive, "r") as z:
        matches = [
            name for name in z.namelist()
            if "deberta" in name.lower()
            or "model.safetensors" in name.lower()
        ]

        print("\n".join(matches[:30]) if matches else
              "No DeBERTa or model.safetensors files found inside.")


ARCHIVE: news-bias-detection-main (2).zip
No DeBERTa or model.safetensors files found inside.

ARCHIVE: news-bias-detection-main UCE2024431 A2.zip
No DeBERTa or model.safetensors files found inside.


In [ ]:
import os

folder = "/content/drive/MyDrive/NewsBais1"

for name in os.listdir(folder):
    print(name)

newsbias_roberta_best
